In [10]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV

In [11]:
live_avg = pd.read_csv("../data/samples/mega_processed/live_avg.csv")
live_avg_norm = pd.read_csv("../data/samples/mega_processed/live_avg_norm.csv")
live_avg_scaled = pd.read_csv("../data/samples/mega_processed/live_avg_scaled.csv")

In [12]:
to_keep = ['GPU Util',
 'Power Draw',
 'GPU Clock',
 'Mem Used',
 'Time']
to_drop = ['Mem Util', 'Time Delta', 'Iter', 'GPU Temp',]

In [13]:
y = live_avg.iloc[:, -2:-1]
y = y['Response Time'].tolist()
live_avg = live_avg.iloc[:, :-1]

In [14]:
live_avg

,GPU Util,Power Draw,GPU Temp,GPU Clock,Mem Used,Mem Util,Time,Time Delta,Iter,Response Time
0,0.000,39.090000,28.000000,1.080000e+09,4.902093e+08,0.028571,-2.208942e+18,4.928571,7.5,8128572456
1,0.770,232.600000,38.000000,1.410000e+09,6.808404e+09,0.400000,-2.208942e+18,12.000000,5.5,5280317649
2,0.770,232.600000,38.000000,1.410000e+09,6.808404e+09,0.390000,-2.208942e+18,19.533333,8.0,8734937890
3,0.490,62.740000,41.000000,1.410000e+09,6.808404e+09,0.250000,-2.208942e+18,24.666667,2.0,1361475752
4,0.490,62.740000,41.000000,1.410000e+09,6.808404e+09,0.250000,-2.208942e+18,27.400000,3.0,2794903187
...,...,...,...,...,...,...,...,...,...,...
402,0.530,73.790000,58.000000,1.410000e+09,6.810501e+09,0.260000,-2.208941e+18,1595.500000,4.5,3809795284
403,0.695,208.304286,61.142857,1.397143e+09,6.810501e+09,0.371429,-2.208941e+18,1601.642857,7.5,7607747587
404,0.740,244.990000,62.000000,1.395000e+09,6.810501e+09,0.390000,-2.208941e+18,1606.000000,1.5,902482509
405,0.740,244.990000,62.000000,1.395000e+09,6.810501e+09,0.390000,-2.208941e+18,1607.666667,2.0,1202432714


In [16]:
live_avg_norm.columns = live_avg.columns[:-1]
live_avg_norm_aug = live_avg_norm.drop(columns=to_drop)
live_avg_norm = live_avg_norm.drop(columns=['Iter', 'Time'])

In [17]:
X_train, X_test, y_train, y_test = train_test_split(live_avg_norm_aug, y, test_size=0.2, shuffle=True, random_state=20)

In [18]:
training_dmatrix = xgb.DMatrix(X_train, y_train)
testing_dmatrix = xgb.DMatrix(X_test, y_test)

In [19]:
GRID_PARAMS = {
    'xgb':{
        'objective':'reg:squarederror',
        'max_depth':4,
        "learning_rate":0.005,
        "device":"cuda",
        "booster":"gbtree",
        "early_stopping_rounds":100,
        "n_estimators":150,
    }
}

In [20]:
regressor = xgb.XGBRegressor(objective='reg:squarederror', max_depth=4, learning_rate=0.005, device='cpu', booster='gbtree', n_estimators=150)

In [ ]:
regressor.fit(X_train, y_train)

In [22]:
regressor.score(X_test, y_test)

0.27026867866516113

In [23]:
dict(zip(live_avg_norm_aug.columns, regressor.feature_importances_))

{'GPU Util': 0.17959802,
 'Power Draw': 0.1623349,
 'GPU Clock': 0.3208715,
 'Mem Used': 0.3371956,
 'Time': 0.0}